# Naive Bayes for Beginners

This notebook teaches Naive Bayes from scratch in simple language.

We will learn:
- what Naive Bayes is
- why it is called "Naive"
- the key formulas
- the main variants of Naive Bayes
- how to train a model with scikit-learn
- how to read the model output

We will use the Iris dataset from scikit-learn because it is small, clean, and good for learning.

In [ ]:
# Import the tools we need
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load a very common beginner dataset
iris = load_iris()

# Features are the input columns
X = iris.data

# Target is the flower type we want to predict
y = iris.target

# Put the data into a table so it is easy to inspect
df = pd.DataFrame(X, columns=iris.feature_names)
df["target"] = y
df["target_name"] = df["target"].map(lambda index: iris.target_names[index])

print("Dataset shape:", df.shape)
print("Feature names:", list(iris.feature_names))
print()
df.head()

## 1. What Is Naive Bayes?

Naive Bayes is a **probabilistic classification algorithm**.

That means:
- it predicts a class such as `setosa`, `versicolor`, or `virginica`
- it does this by using **probabilities**
- it asks: "Given these feature values, which class is most likely?"

The word **Bayes** comes from **Bayes' theorem**.

The word **Naive** means the algorithm makes a very strong simplifying assumption:
- it assumes the features are conditionally independent once the class is known

In real life, features are often related to each other, so this assumption is not fully true.
But even with that simplification, Naive Bayes often works surprisingly well.

Common uses:
- text classification such as spam detection
- sentiment analysis
- simple medical diagnosis tasks
- fast baseline models in machine learning

## 2. The Core Formula

Bayes' theorem says:

$$P(C \mid X) = \frac{P(X \mid C) \cdot P(C)}{P(X)}$$

Meaning of each term:
- $C$: a class, for example `setosa`
- $X$: the observed feature values
- $P(C \mid X)$: posterior probability, the probability of class $C$ after seeing the data
- $P(X \mid C)$: likelihood, the probability of seeing the features if the class is $C$
- $P(C)$: prior probability of class $C$
- $P(X)$: evidence, the total probability of the observed features

Naive Bayes adds the independence assumption:

$$P(X \mid C) = P(x_1, x_2, ..., x_n \mid C) \approx \prod_{i=1}^{n} P(x_i \mid C)$$

So the classifier becomes:

$$P(C \mid X) \propto P(C) \prod_{i=1}^{n} P(x_i \mid C)$$

The model predicts the class with the largest posterior probability.

## 3. Variants of Naive Bayes and the Gaussian Formula

Different Naive Bayes models make different assumptions about the feature values.

### Gaussian Naive Bayes
Use this when features are continuous numbers, such as height, weight, or flower measurements.

It assumes each feature follows a Gaussian distribution inside each class:

$$P(x_i \mid C) = \frac{1}{\sqrt{2\pi\sigma_C^2}} \exp\left(-\frac{(x_i - \mu_C)^2}{2\sigma_C^2}\right)$$

Where:
- $\mu_C$ is the mean of the feature for class $C$
- $\sigma_C^2$ is the variance of the feature for class $C$

### Other common types
- **MultinomialNB**: good for word counts and text data
- **BernoulliNB**: good for binary features such as 0 or 1
- **CategoricalNB**: good for categorical inputs

For the Iris dataset, **GaussianNB** is the right choice because the features are continuous measurements.

In [ ]:
# Split the data into training and testing parts
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
    # stratify keeps the class balance similar in train and test

)

# Create the model
model = GaussianNB()

# Learn the pattern from the training data
model.fit(X_train, y_train)

# Use the trained model to predict the test labels
y_pred = model.predict(X_test)

# Measure how well the model performed
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=iris.target_names)

print(f"Accuracy: {accuracy:.3f}")
print()
print("Confusion matrix:")
print(cm)
print()
print("Classification report:")
print(report)

In [ ]:
# Look at what the model learned from the training data
print("Class names:", iris.target_names)
print()

# Prior probabilities P(C) learned from the training set
print("Class prior probabilities:")
print(model.class_prior_)
print()

# Mean of each feature inside each class
print("Feature means for each class (theta_):")
print(pd.DataFrame(model.theta_, columns=iris.feature_names, index=iris.target_names))
print()

# Variance of each feature inside each class
print("Feature variances for each class (var_):")
print(pd.DataFrame(model.var_, columns=iris.feature_names, index=iris.target_names))
print()

# Predicted probabilities for the first 5 test examples
probabilities = model.predict_proba(X_test[:5])
probability_table = pd.DataFrame(probabilities, columns=iris.target_names)
print("Predicted probabilities for the first 5 test rows:")
probability_table

## 4. How to Read the Results

### Accuracy
Accuracy tells us the fraction of correct predictions:

$$\text{Accuracy} = \frac{\text{Number of correct predictions}}{\text{Total number of predictions}}$$

### Confusion matrix
The confusion matrix shows which classes were predicted correctly and where mistakes happened.
- diagonal values are correct predictions
- off-diagonal values are mistakes

### Classification report
This report shows:
- **precision**: when the model predicts a class, how often it is right
- **recall**: out of all true examples of a class, how many it found
- **f1-score**: a balance between precision and recall

### Learned parameters
- `class_prior_` stores $P(C)$
- `theta_` stores the mean values $\mu$
- `var_` stores the variances $\sigma^2$

These are exactly the numbers the Gaussian formula uses during prediction.

In [ ]:
# Make a prediction for one new flower measurement
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])

predicted_class_index = model.predict(new_flower)[0]
predicted_probabilities = model.predict_proba(new_flower)[0]

print("New flower measurements:", new_flower[0])
print("Predicted class:", iris.target_names[predicted_class_index])
print()
print("Class probabilities:")

for class_name, probability in zip(iris.target_names, predicted_probabilities):
    print(f"{class_name}: {probability:.4f}")

## 5. Strengths, Weaknesses, and Final Summary

### Strengths
- very fast to train
- very fast to predict
- works well on many small and medium problems
- excellent baseline model
- especially strong for text classification

### Weaknesses
- the independence assumption is often unrealistic
- performance may drop when features are strongly correlated
- probability estimates can be overconfident
- GaussianNB assumes a Gaussian-like distribution for each feature inside each class

### Final idea to remember
Naive Bayes predicts the class with the highest value of:

$$P(C) \prod_{i=1}^{n} P(x_i \mid C)$$

So the algorithm combines:
- what was common before seeing the sample, and
- how likely the sample features are under each class

If you understand that sentence, you understand the heart of Naive Bayes.